# Entraînement EfficientNetB0 pour la pneumonie

Ce notebook entraîne un modèle **EfficientNetB0** (transfer learning) pour classer les radios thoraciques en trois classes : **normal**, **bacteria**, **virus**.

## Choix des outils

- **PyTorch** : j'utilise PyTorch (et non Keras / TensorFlow ou scikit‑learn) car il est aujourd'hui très répandu pour la recherche en deep learning, offre un contrôle fin sur la boucle d'entraînement et s'intègre naturellement avec les DataLoaders et les transformations avancées (Albumentations).
- **EfficientNetB0 (torchvision)** : EfficientNet est une famille de réseaux de classification d'images efficaces et performants. La version B0 est légère, ce qui est adapté à un projet pédagogique / expérimental tout en restant suffisamment puissante pour de la classification médicale simple.
- **Transfer learning** : on part d'un modèle pré‑entraîné sur ImageNet (grande base d'images naturelles) et on ne remplace que la couche de classification finale pour apprendre nos 3 classes de pneumonie. Cela permet de converger plus vite et de mieux généraliser avec un dataset médical de taille modérée.

Les loaders de données et les augmentations sont définis dans `on_the_fly_augmentation.ipynb` et seront exécutés ici via `%run`. Le notebook d'augmentation utilise **Albumentations** pour des transformations "médicalement sûres" et la bibliothèque **datasets** de Hugging Face pour charger les radios thoraciques.

## 1. Chargement du pipeline d'augmentation

Dans cette première étape, on **réutilise** le notebook `on_the_fly_augmentation.ipynb` :
- il télécharge le jeu de données de radios thoraciques,
- applique les augmentations définies avec Albumentations,
- construit `train_loader`, `val_loader` et `test_loader` (objets PyTorch `DataLoader`).

Ainsi, ce notebook se concentre uniquement sur la partie **modèle + entraînement**.

In [ ]:
# Exécuter le notebook d’augmentation on-the-fly
# Ceci définit : dataset, train_data, val_data, test_data,
# ainsi que train_loader, val_loader, test_loader

%run "../on_the_fly_augmentation.ipynb"

## 2. Définition du modèle EfficientNetB0 et des hyperparamètres

Ici, on :
- détecte le **device** disponible (`cuda` si un GPU est présent, sinon `cpu`),
- fixe quelques hyperparamètres simples (nombre d'époques, taux d'apprentissage, nombre de classes),
- charge un backbone **EfficientNetB0** pré‑entraîné sur ImageNet depuis `torchvision`,
- remplace la **couche de classification finale** par une couche linéaire avec 3 sorties correspondant à nos classes,
- définit la fonction de coût (**CrossEntropyLoss**) adaptée à la classification multi‑classe,
- configure l'optimiseur **Adam** pour mettre à jour les poids du réseau.

In [ ]:
# Exécuter le notebook d’augmentation on-the-fly
# Ceci définit : dataset, train_data, val_data, test_data,
# ainsi que train_loader, val_loader, test_loader

%run "../on_the_fly_augmentation.ipynb"

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# Détection du device (GPU si disponible, sinon CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device utilisé : {device}")

# Hyperparamètres de base pour l'entraînement
NUM_CLASSES = 3  # normal, bacteria, virus
learning_rate = 1e-4
num_epochs = 10

# Chargement du backbone EfficientNetB0 pré-entraîné sur ImageNet
weights = EfficientNet_B0_Weights.IMAGENET1K_V1
model = efficientnet_b0(weights=weights)

# Remplacement de la couche de classification finale
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)

model = model.to(device)

# Fonction de coût adaptée à la classification multi-classe
criterion = nn.CrossEntropyLoss()

# Optimiseur Adam classique pour l'ensemble des paramètres du modèle
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

print("Modèle EfficientNetB0 initialisé pour 3 classes.")

## 3. Boucle d'entraînement et de validation

On parcourt plusieurs fois le jeu d'entraînement (**époques**) et, à chaque itération :
- on met le modèle en mode `train()` pour autoriser la mise à jour des poids,
- on lit un batch d'images et de labels depuis `train_loader`,
- on calcule la prédiction, la loss, puis on fait un **backward** + **optimizer.step()`,
- on agrège la loss moyenne sur tout le jeu d'entraînement.

Ensuite, on met le modèle en mode `eval()` pour la **validation** :
- on désactive le gradient avec `torch.no_grad()`,
- on calcule la loss de validation,
- on mesure aussi une première métrique simple : l'**accuracy** sur le jeu de validation.

Les valeurs de loss / accuracy par époque sont stockées dans `train_history` pour pouvoir les tracer ensuite.

In [ ]:
from tqdm.auto import tqdm

train_history = {"train_loss": [], "val_loss": [], "val_acc": []}

for epoch in range(1, num_epochs + 1):
    print(f"\n===== Epoch {epoch}/{num_epochs} =====")

    # --------------------
    # Phase entraînement
    # --------------------
    model.train()
    running_loss = 0.0

    for batch in tqdm(train_loader, desc="Train", leave=False):
        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_train_loss = running_loss / len(train_loader.dataset)

    # --------------------
    # Phase validation
    # --------------------
    model.eval()
    val_running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Val", leave=False):
            images = batch["image"].to(device)
            labels = batch["label"].to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    epoch_val_loss = val_running_loss / len(val_loader.dataset)
    epoch_val_acc = correct / total if total > 0 else 0.0

    train_history["train_loss"].append(epoch_train_loss)
    train_history["val_loss"].append(epoch_val_loss)
    train_history["val_acc"].append(epoch_val_acc)

    print(f"Train loss: {epoch_train_loss:.4f} | Val loss: {epoch_val_loss:.4f} | Val acc: {epoch_val_acc:.4f}")

## 4. Sauvegarde du modèle et visualisation des courbes

À la fin de l'entraînement :
- on **sauvegarde** les poids du modèle dans un fichier `efficientnet_b0_pneumonia.pt` (format PyTorch),
- on trace les courbes de **loss** (train / validation) et d'**accuracy de validation** pour vérifier rapidement si le modèle converge et s'il y a du surapprentissage (overfitting).

In [ ]:
import matplotlib.pyplot as plt

# Sauvegarde du modèle
model_path = "./efficientnet_b0_pneumonia.pt"
torch.save(model.state_dict(), model_path)
print(f"Modèle sauvegardé dans {model_path}")

# Affichage simple des courbes de loss et accuracy
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(train_history["train_loss"], label="Train loss")
plt.plot(train_history["val_loss"], label="Val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Évolution de la loss")

plt.subplot(1, 2, 2)
plt.plot(train_history["val_acc"], label="Val acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Accuracy validation")

plt.tight_layout()
plt.show()